In [1]:
import os

os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.16.8-hotspot"
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PYSPARK_PYTHON"] = r"C:\Users\shivn\DE-Interview-Prep\.venv\Scripts\python.exe"
os.environ["PYSPARK_DRIVER_PYTHON"] = r"C:\Users\shivn\DE-Interview-Prep\.venv\Scripts\python.exe"

# NEW LINE — add hadoop bin to PATH
os.environ["PATH"] = r"C:\hadoop\bin" + ";" + os.environ["PATH"]

if "SPARK_HOME" in os.environ:
    del os.environ["SPARK_HOME"]

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("BTS Flight Delay - Q1 2024") \
    .master("local[*]") \
    .getOrCreate()

print("Spark is running!")
print(f"Spark version: {spark.version}")

Spark is running!
Spark version: 3.5.1


In [2]:
jan = spark.read.csv(r"C:\Users\shivn\Downloads\Practice Data 3 months\January_2024.csv",header=True, inferSchema=True)
feb = spark.read.csv(r"C:\Users\shivn\Downloads\Practice Data 3 months\February_2024.csv",header=True, inferSchema=True)
mar = spark.read.csv(r"C:\Users\shivn\Downloads\Practice Data 3 months\March_2024.csv", header=True, inferSchema=True)

df = jan.unionByName(feb).unionByName(mar)

print(f"January rows:  {jan.count():,}")
print(f"February rows: {feb.count():,}")
print(f"March rows:    {mar.count():,}")
print(f"Total Q1 2024: {df.count():,}")
print(f"Total columns: {len(df.columns)}")

January rows:  547,271
February rows: 519,221
March rows:    591,767
Total Q1 2024: 1,658,259
Total columns: 37


In [3]:
df.show(5)

+----+-----+------------+-----------+--------------------+-----------------+--------+-----------------+------+----------------+----------------+----+--------------+--------------+------------+--------+---------+-------------+---------+------------+--------+---------+-------------+---------+---------+-----------------+--------+----------------+-------------------+--------+-------+--------+-------------+-------------+---------+--------------+-------------------+
|YEAR|MONTH|DAY_OF_MONTH|DAY_OF_WEEK|             FL_DATE|OP_UNIQUE_CARRIER|TAIL_NUM|OP_CARRIER_FL_NUM|ORIGIN|ORIGIN_CITY_NAME|ORIGIN_STATE_ABR|DEST|DEST_CITY_NAME|DEST_STATE_ABR|CRS_DEP_TIME|DEP_TIME|DEP_DELAY|DEP_DELAY_NEW|DEP_DEL15|CRS_ARR_TIME|ARR_TIME|ARR_DELAY|ARR_DELAY_NEW|ARR_DEL15|CANCELLED|CANCELLATION_CODE|DIVERTED|CRS_ELAPSED_TIME|ACTUAL_ELAPSED_TIME|AIR_TIME|FLIGHTS|DISTANCE|CARRIER_DELAY|WEATHER_DELAY|NAS_DELAY|SECURITY_DELAY|LATE_AIRCRAFT_DELAY|
+----+-----+------------+-----------+--------------------+------------

In [4]:
df.show(5,truncate=False,vertical=True)

-RECORD 0-----------------------------------
 YEAR                | 2024                 
 MONTH               | 1                    
 DAY_OF_MONTH        | 1                    
 DAY_OF_WEEK         | 1                    
 FL_DATE             | 1/1/2024 12:00:00 AM 
 OP_UNIQUE_CARRIER   | 9E                   
 TAIL_NUM            | N131EV               
 OP_CARRIER_FL_NUM   | 5225                 
 ORIGIN              | ATL                  
 ORIGIN_CITY_NAME    | Atlanta, GA          
 ORIGIN_STATE_ABR    | GA                   
 DEST                | AVL                  
 DEST_CITY_NAME      | Asheville, NC        
 DEST_STATE_ABR      | NC                   
 CRS_DEP_TIME        | 1410                 
 DEP_TIME            | 1406                 
 DEP_DELAY           | -4.0                 
 DEP_DELAY_NEW       | 0.0                  
 DEP_DEL15           | 0.0                  
 CRS_ARR_TIME        | 1511                 
 ARR_TIME            | 1454                 
 ARR_DELAY

In [5]:
# Quick data health check
print(f"Total rows: {df.count():,}")
print(f"Total columns: {len(df.columns)}")
print(f"\nNull counts per column:")
from pyspark.sql.functions import col, sum as spark_sum
df.select([spark_sum(col(c).isNull().cast("int")).alias(c) 
           for c in df.columns]).show(vertical=True)

Total rows: 1,658,259
Total columns: 37

Null counts per column:
-RECORD 0----------------------
 YEAR                | 0       
 MONTH               | 0       
 DAY_OF_MONTH        | 0       
 DAY_OF_WEEK         | 0       
 FL_DATE             | 0       
 OP_UNIQUE_CARRIER   | 0       
 TAIL_NUM            | 6570    
 OP_CARRIER_FL_NUM   | 0       
 ORIGIN              | 0       
 ORIGIN_CITY_NAME    | 0       
 ORIGIN_STATE_ABR    | 0       
 DEST                | 0       
 DEST_CITY_NAME      | 0       
 DEST_STATE_ABR      | 0       
 CRS_DEP_TIME        | 0       
 DEP_TIME            | 27557   
 DEP_DELAY           | 27679   
 DEP_DELAY_NEW       | 27679   
 DEP_DEL15           | 27679   
 CRS_ARR_TIME        | 0       
 ARR_TIME            | 29028   
 ARR_DELAY           | 32207   
 ARR_DELAY_NEW       | 32207   
 ARR_DEL15           | 32207   
 CANCELLED           | 0       
 CANCELLATION_CODE   | 1629748 
 DIVERTED            | 0       
 CRS_ELAPSED_TIME    | 1       
 ACTUAL

In [6]:
# Verify the %  NULL pattern
total_rows = 1_658_259
delay_cause_nulls = 1_327_781
null_percentage = delay_cause_nulls / total_rows * 100
print(f"Delay cause NULL %: {null_percentage:.1f}%")

Delay cause NULL %: 80.1%


In [7]:
# Verify: when ARR_DEL15 = 0, are delay columns always NULL?

df.filter(
    (col("ARR_DEL15") == 0) & 
    (col("CARRIER_DELAY").isNotNull())
).count()

# If this returns 0 → rule confirmed
# If this returns > 0 → rule violated, investigate

0

**DAY_1**

In [8]:
# PROBLEM 1 — Basic DataFrame Operations
# 1. Print the schema (column names + data types)
# 2. Select only these 5 columns:
#    OP_UNIQUE_CARRIER, ORIGIN, DEST,
#    DEP_DELAY, ARR_DELAY
# 3. Show 5 rows of this smaller DataFrame

df.printSchema()
df.select("OP_UNIQUE_CARRIER", "ORIGIN", "DEST", "DEP_DELAY", "ARR_DELAY").show(5)

root
 |-- YEAR: integer (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- DAY_OF_MONTH: integer (nullable = true)
 |-- DAY_OF_WEEK: integer (nullable = true)
 |-- FL_DATE: string (nullable = true)
 |-- OP_UNIQUE_CARRIER: string (nullable = true)
 |-- TAIL_NUM: string (nullable = true)
 |-- OP_CARRIER_FL_NUM: integer (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- ORIGIN_CITY_NAME: string (nullable = true)
 |-- ORIGIN_STATE_ABR: string (nullable = true)
 |-- DEST: string (nullable = true)
 |-- DEST_CITY_NAME: string (nullable = true)
 |-- DEST_STATE_ABR: string (nullable = true)
 |-- CRS_DEP_TIME: integer (nullable = true)
 |-- DEP_TIME: integer (nullable = true)
 |-- DEP_DELAY: double (nullable = true)
 |-- DEP_DELAY_NEW: double (nullable = true)
 |-- DEP_DEL15: double (nullable = true)
 |-- CRS_ARR_TIME: integer (nullable = true)
 |-- ARR_TIME: integer (nullable = true)
 |-- ARR_DELAY: double (nullable = true)
 |-- ARR_DELAY_NEW: double (nullable = true)
 |-- A

In [9]:
# PROBLEM 2 — Filtering
# Using your BTS DataFrame:
#
# 1. Filter only DELAYED flights (ARR_DEL15 = 1)
# 2. From those, filter flights where ARR_DELAY > 60
#    (delayed more than 1 hour)
# 3. Select these columns:
#    OP_UNIQUE_CARRIER, ORIGIN, DEST, ARR_DELAY
# 4. Show 10 rows
# 5. Print how many flights were delayed more than 1 hour
#
# Hint: df.filter(col("column") == value)
#       You can chain .filter().filter()
#       OR combine with & operator


delayed = df.filter((col("ARR_DEL15") == 1) & (col("ARR_DELAY") > 60)).select("OP_UNIQUE_CARRIER", "ORIGIN", "DEST", "ARR_DELAY")
delayed.show(10)
print(f"Flights delayed more than 1 hour: {delayed.count():,}")


+-----------------+------+----+---------+
|OP_UNIQUE_CARRIER|ORIGIN|DEST|ARR_DELAY|
+-----------------+------+----+---------+
|               9E|   AVL| ATL|    207.0|
|               9E|   CLT| JFK|    162.0|
|               9E|   RDU| LGA|    204.0|
|               AA|   SAT| CLT|     78.0|
|               AA|   BOS| LAX|     92.0|
|               AA|   AUS| PHX|    108.0|
|               AA|   PHX| PDX|     61.0|
|               AA|   DFW| AUS|    121.0|
|               AA|   PHL| CLT|     83.0|
|               AA|   CLT| PHL|     91.0|
+-----------------+------+----+---------+
only showing top 10 rows

Flights delayed more than 1 hour: 110,406


In [10]:
# PROBLEM 3 — Aggregations
# Using your BTS DataFrame:
#
# 1. Find the TOP 5 carriers by:
#    → total number of flights
#    → average arrival delay (all flights, not just delayed)
#    → total flights delayed more than 15 mins
#
# 2. Sort by total flights descending
#
# 3. Show the result
#
# Hint:
# from pyspark.sql.functions import count, avg, sum, round
# df.groupBy("column").agg(
#     count("*").alias("total_flights"),
#     avg("column").alias("avg_delay")
# )


from pyspark.sql.functions import count, avg, round, when

top_carriers = df.groupBy("OP_UNIQUE_CARRIER") \
    .agg(
        count("*").alias("total_flights"),
        round(avg("ARR_DELAY"), 2).alias("avg_arr_delay"),
        count(when(col("ARR_DEL15") == 1, 1)).alias("total_delayed")
    ) \
    .orderBy("total_flights", ascending=False) \
    .limit(5)

top_carriers.show()

+-----------------+-------------+-------------+-------------+
|OP_UNIQUE_CARRIER|total_flights|avg_arr_delay|total_delayed|
+-----------------+-------------+-------------+-------------+
|               WN|       345868|          4.8|        71490|
|               AA|       234475|        13.64|        58234|
|               DL|       228270|         0.82|        35437|
|               UA|       180618|         4.04|        32815|
|               OO|       166380|         7.23|        30987|
+-----------------+-------------+-------------+-------------+



**Practice_Problems**

In [11]:
df.show(5)

+----+-----+------------+-----------+--------------------+-----------------+--------+-----------------+------+----------------+----------------+----+--------------+--------------+------------+--------+---------+-------------+---------+------------+--------+---------+-------------+---------+---------+-----------------+--------+----------------+-------------------+--------+-------+--------+-------------+-------------+---------+--------------+-------------------+
|YEAR|MONTH|DAY_OF_MONTH|DAY_OF_WEEK|             FL_DATE|OP_UNIQUE_CARRIER|TAIL_NUM|OP_CARRIER_FL_NUM|ORIGIN|ORIGIN_CITY_NAME|ORIGIN_STATE_ABR|DEST|DEST_CITY_NAME|DEST_STATE_ABR|CRS_DEP_TIME|DEP_TIME|DEP_DELAY|DEP_DELAY_NEW|DEP_DEL15|CRS_ARR_TIME|ARR_TIME|ARR_DELAY|ARR_DELAY_NEW|ARR_DEL15|CANCELLED|CANCELLATION_CODE|DIVERTED|CRS_ELAPSED_TIME|ACTUAL_ELAPSED_TIME|AIR_TIME|FLIGHTS|DISTANCE|CARRIER_DELAY|WEATHER_DELAY|NAS_DELAY|SECURITY_DELAY|LATE_AIRCRAFT_DELAY|
+----+-----+------------+-----------+--------------------+------------

In [12]:
df.columns

['YEAR',
 'MONTH',
 'DAY_OF_MONTH',
 'DAY_OF_WEEK',
 'FL_DATE',
 'OP_UNIQUE_CARRIER',
 'TAIL_NUM',
 'OP_CARRIER_FL_NUM',
 'ORIGIN',
 'ORIGIN_CITY_NAME',
 'ORIGIN_STATE_ABR',
 'DEST',
 'DEST_CITY_NAME',
 'DEST_STATE_ABR',
 'CRS_DEP_TIME',
 'DEP_TIME',
 'DEP_DELAY',
 'DEP_DELAY_NEW',
 'DEP_DEL15',
 'CRS_ARR_TIME',
 'ARR_TIME',
 'ARR_DELAY',
 'ARR_DELAY_NEW',
 'ARR_DEL15',
 'CANCELLED',
 'CANCELLATION_CODE',
 'DIVERTED',
 'CRS_ELAPSED_TIME',
 'ACTUAL_ELAPSED_TIME',
 'AIR_TIME',
 'FLIGHTS',
 'DISTANCE',
 'CARRIER_DELAY',
 'WEATHER_DELAY',
 'NAS_DELAY',
 'SECURITY_DELAY',
 'LATE_AIRCRAFT_DELAY']

In [13]:
# Now we do this problems. 

# Q1 — Filter + Count
# Find all flights that were CANCELLED (CANCELLED = 1)
# From those, how many were cancelled per carrier?
# Sort by cancellations descending.
# Show top 5.
# Which carrier cancelled the most flights in Q1 2024?

# df.columns
cancelled = df.filter(df["Cancelled"]==1).groupBy("OP_UNIQUE_CARRIER").count().orderBy("count", ascending = False)
cancelled.show(5)


+-----------------+-----+
|OP_UNIQUE_CARRIER|count|
+-----------------+-----+
|               UA| 5382|
|               WN| 5202|
|               OO| 3864|
|               AS| 3399|
|               AA| 2265|
+-----------------+-----+
only showing top 5 rows



In [14]:
# Q2 — Filter + Select
# Find all flights where:
#   → Origin is "ATL" (Atlanta)
#   → AND flight was delayed more than 30 minutes (ARR_DELAY > 30)
# Select: OP_UNIQUE_CARRIER, DEST, ARR_DELAY, FL_DATE
# Show 10 rows.
# How many total ATL flights were delayed more than 30 mins?

flights = df.filter((df["ORIGIN"] == "ATL") & (df["ARR_DELAY"] > 30)).select("OP_UNIQUE_CARRIER", "DEST", "ARR_DELAY", "FL_DATE")
flights.show(10)

+-----------------+----+---------+--------------------+
|OP_UNIQUE_CARRIER|DEST|ARR_DELAY|             FL_DATE|
+-----------------+----+---------+--------------------+
|               AA| DFW|     53.0|1/1/2024 12:00:00 AM|
|               B6| FLL|     33.0|1/1/2024 12:00:00 AM|
|               DL| MKE|    100.0|1/1/2024 12:00:00 AM|
|               DL| LAX|     39.0|1/1/2024 12:00:00 AM|
|               DL| MCO|     37.0|1/1/2024 12:00:00 AM|
|               DL| MSY|     34.0|1/1/2024 12:00:00 AM|
|               DL| MCO|     32.0|1/1/2024 12:00:00 AM|
|               DL| JAX|     44.0|1/1/2024 12:00:00 AM|
|               DL| GEG|     40.0|1/1/2024 12:00:00 AM|
|               DL| RSW|     50.0|1/1/2024 12:00:00 AM|
+-----------------+----+---------+--------------------+
only showing top 10 rows



In [15]:
# Q3 — Aggregation
# For each ORIGIN airport:
#   → Count total flights
#   → Calculate average departure delay
#   → Count how many flights departed early (DEP_DELAY < 0)
# Show TOP 10 busiest origin airports by total flights
# Which airport has the best on-time departure performance?

from pyspark.sql.functions import count, avg, col, sum, when

Airport_Stats = df.groupBy("ORIGIN").agg(count("*").alias("Total_Flights"),avg("Dep_Delay").alias("Avg_dep_delay"),
                                  sum(when(col("Dep_Delay")<0, 1).otherwise(0)).alias("Early_departures")).orderBy("Total_Flights",ascending=False)

Airport_Stats.show(10)

+------+-------------+------------------+----------------+
|ORIGIN|Total_Flights|     Avg_dep_delay|Early_departures|
+------+-------------+------------------+----------------+
|   ATL|        79595| 8.718639575971732|           48685|
|   DFW|        71816| 16.22775368358597|           39283|
|   DEN|        70096|13.924102054534297|           32927|
|   ORD|        62162|12.574661454467481|           37028|
|   CLT|        49546| 13.43166798836902|           27259|
|   PHX|        47576|10.281152598435215|           26615|
|   LAX|        45298|10.643948374974837|           26982|
|   LAS|        45074|13.150637780927314|           23026|
|   MCO|        43523| 16.45995471416233|           21304|
|   LGA|        38710| 9.631565035815294|           25455|
+------+-------------+------------------+----------------+
only showing top 10 rows



In [16]:
# Q4 — Schema Investigation
# FL_DATE is currently a string type.
# 1. Print 5 unique values of FL_DATE to see the format
# 2. Count how many unique flight dates exist in Q1 2024
# 3. How many flights operated on January 15th 2024?
# Hint: df.select("FL_DATE").distinct()
#       df.filter(col("FL_DATE").contains("1/15/2024"))


# 1. See 5 unique FL_DATE values
df.select("FL_DATE").distinct().show(5)

# 2. Count unique flight dates
print(df.select("FL_DATE").distinct().count())

# 3. Flights on January 15th
jan15 = df.filter(col("FL_DATE").contains("1/15/2024"))
print(f"Flights on Jan 15: {jan15.count():,}")

+--------------------+
|             FL_DATE|
+--------------------+
|1/1/2024 12:00:00 AM|
|1/2/2024 12:00:00 AM|
|1/3/2024 12:00:00 AM|
|1/4/2024 12:00:00 AM|
|1/6/2024 12:00:00 AM|
+--------------------+
only showing top 5 rows

91
Flights on Jan 15: 18,620


In [17]:
# Q5 — Combined (hardest)
# Find the TOP 5 ROUTES (origin → destination pairs)
# with the highest average arrival delay
# BUT only include routes that had MORE than 100 flights
# (to avoid single-flight anomalies skewing the average)
# Show: ORIGIN, DEST, total_flights, avg_arr_delay
# Sort by avg_arr_delay descending
# Hint: .filter(col("total_flights") > 100) after groupBy
#       This is called HAVING in SQL

top_routes = df.groupBy("ORIGIN", "DEST").agg(count("*").alias("Total_Flights"), avg("ARR_DELAY").alias("avg_arrival_delay")).\
    filter(col("Total_Flights")>100).orderBy("avg_arrival_delay",ascending=False)
top_routes.show(10)

+------+----+-------------+------------------+
|ORIGIN|DEST|Total_Flights| avg_arrival_delay|
+------+----+-------------+------------------+
|   EYW| DFW|          182| 76.45454545454545|
|   EGE| MIA|          104|             65.91|
|   RDM| SFO|          253| 63.33613445378151|
|   STX| MIA|          182| 59.81666666666667|
|   JAC| DFW|          176| 57.02840909090909|
|   SFO| ASE|          182|54.858024691358025|
|   SJU| BDL|          254|51.275720164609055|
|   SAN| MIA|          181| 49.85875706214689|
|   ASE| ATL|          182|49.239263803680984|
|   ASE| SFO|          181|47.524096385542165|
+------+----+-------------+------------------+
only showing top 10 rows



**DAY_2**

In [18]:
# Find all flights where departure delay is more than 60 minutes.
# Show 5 rows. How many such flights exist in Q1 2024?

delayed = df.filter(df["DEP_DELAY"] > 60)
delayed.show(5)
print(f"Total Flights delayed > 60 mins: {delayed.count():,}")

+----+-----+------------+-----------+--------------------+-----------------+--------+-----------------+------+------------------+----------------+----+---------------+--------------+------------+--------+---------+-------------+---------+------------+--------+---------+-------------+---------+---------+-----------------+--------+----------------+-------------------+--------+-------+--------+-------------+-------------+---------+--------------+-------------------+
|YEAR|MONTH|DAY_OF_MONTH|DAY_OF_WEEK|             FL_DATE|OP_UNIQUE_CARRIER|TAIL_NUM|OP_CARRIER_FL_NUM|ORIGIN|  ORIGIN_CITY_NAME|ORIGIN_STATE_ABR|DEST| DEST_CITY_NAME|DEST_STATE_ABR|CRS_DEP_TIME|DEP_TIME|DEP_DELAY|DEP_DELAY_NEW|DEP_DEL15|CRS_ARR_TIME|ARR_TIME|ARR_DELAY|ARR_DELAY_NEW|ARR_DEL15|CANCELLED|CANCELLATION_CODE|DIVERTED|CRS_ELAPSED_TIME|ACTUAL_ELAPSED_TIME|AIR_TIME|FLIGHTS|DISTANCE|CARRIER_DELAY|WEATHER_DELAY|NAS_DELAY|SECURITY_DELAY|LATE_AIRCRAFT_DELAY|
+----+-----+------------+-----------+--------------------+------

In [19]:
# Find flights where:
#   → Carrier is "DL" (Delta)
#   AND → ARR_DELAY > 45 minutes
#   OR  → DEP_DELAY > 45 minutes

# How many such flights exist?
# Show 5 rows with columns: OP_UNIQUE_CARRIER, ORIGIN, DEST, DEP_DELAY, ARR_DELAY

flghts = df.filter((df["OP_UNIQUE_CARRIER"] == "DL") & ((df["ARR_DELAY"]>45) | (df["DEP_DELAY"] > 45))).\
    select("OP_UNIQUE_CARRIER", "ORIGIN", "DEST", "DEP_DELAY", "ARR_DELAY")
flghts.show(5)
print(f"Total Count: {flghts.count():,}")

+-----------------+------+----+---------+---------+
|OP_UNIQUE_CARRIER|ORIGIN|DEST|DEP_DELAY|ARR_DELAY|
+-----------------+------+----+---------+---------+
|               DL|   BOS| PBI|     87.0|     57.0|
|               DL|   SAN| DTW|    123.0|    109.0|
|               DL|   BOS| JAX|     59.0|     41.0|
|               DL|   LAS| DTW|     96.0|     87.0|
|               DL|   ONT| SEA|     42.0|     49.0|
+-----------------+------+----+---------+---------+
only showing top 5 rows

Total Count: 17,093


In [20]:
# 1. Count how many nulls exist in ARR_DELAY column
# 2. Drop all rows where ARR_DELAY is null → count remaining rows
# 3. Fill nulls in ARR_DELAY with 0 → verify no nulls remain
# 4. Which strategy makes more sense for BTS data and why?

nulls = df.filter(col("ARR_Delay").isNull()).count()
print(f"Null Values : {nulls:,}")

dropped = df.dropna(subset=["ARR_DELAY"])
print(f"Rows Before dropping nulls: {df.count():,}")
print(f"Rows after dropping nulls: {dropped.count():,}")

filling_nulls = df.fillna(0,subset=["ARR_DELAY"])
nulls_after_fill = filling_nulls.filter(col("ARR_DELAY").isNull()).count()
print(f"Nulls remaining after fill: {nulls_after_fill}")

Null Values : 32,207
Rows Before dropping nulls: 1,658,259
Rows after dropping nulls: 1,626,052
Nulls remaining after fill: 0


**Practice_Problems**

In [21]:
# Find all flights that were DIVERTED (DIVERTED = 1)
# Select: OP_UNIQUE_CARRIER, ORIGIN, DEST, FL_DATE
# How many diverted flights happened in Q1 2024?

diverted = df.filter(df["DIVERTED"] == 1) \
             .select("OP_UNIQUE_CARRIER", "ORIGIN", "DEST", "FL_DATE")

diverted.cache()  # store result so Spark doesn't recompute
diverted.show(5)
print(f"The No.Of Flights Diverted: {diverted.count():,}")
diverted.unpersist()  # release memory after done

+-----------------+------+----+--------------------+
|OP_UNIQUE_CARRIER|ORIGIN|DEST|             FL_DATE|
+-----------------+------+----+--------------------+
|               G4|   LAS| GEG|1/1/2024 12:00:00 AM|
|               OO|   SLC| RDM|1/1/2024 12:00:00 AM|
|               OO|   LAX| ASE|1/1/2024 12:00:00 AM|
|               OO|   PHX| RDM|1/1/2024 12:00:00 AM|
|               UA|   ORD| ANC|1/1/2024 12:00:00 AM|
+-----------------+------+----+--------------------+
only showing top 5 rows

The No.Of Flights Diverted: 3,696


DataFrame[OP_UNIQUE_CARRIER: string, ORIGIN: string, DEST: string, FL_DATE: string]

In [22]:
# Find flights where:
#   → Distance > 1000 miles
#   AND → DEP_DELAY < 0 (departed early)
# How many such flights exist?
# Show 5 rows with: ORIGIN, DEST, DISTANCE, DEP_DELAY

distance_flights = df.filter((df["DISTANCE"]>1000) & (df["DEP_DELAY"]<0)).select("ORIGIN", "DEST", "DISTANCE", "DEP_DELAY")
distance_flights.cache()
distance_flights.show(5)
print(f"The Number of flights: {distance_flights.count():,}")
distance_flights.unpersist()

+------+----+--------+---------+
|ORIGIN|DEST|DISTANCE|DEP_DELAY|
+------+----+--------+---------+
|   MCI| LGA|  1107.0|     -5.0|
|   MCI| JFK|  1113.0|     -9.0|
|   LGA| MCI|  1107.0|     -5.0|
|   JFK| MCI|  1113.0|     -5.0|
|   LGA| PNS|  1030.0|     -9.0|
+------+----+--------+---------+
only showing top 5 rows

The Number of flights: 268,170


DataFrame[ORIGIN: string, DEST: string, DISTANCE: double, DEP_DELAY: double]

In [23]:
# Check nulls in these 3 columns:
#   → DEP_DELAY
#   → CARRIER_DELAY
#   → CANCELLATION_CODE
# Which column has the most nulls and why does that make sense?

null_values = df.select(sum(col("DEP_DELAY").isNull().cast("int")).alias("DEP_DELAY_Nulls"),\
                        sum(col("CARRIER_DELAY").isNull().cast("int")).alias("CARRIER_DELAY_Nulls"),\
                        sum(col("CANCELLATION_CODE").isNull().cast("int")).alias("CANCELLATION_CODE"))
null_values.show(5)


+---------------+-------------------+-----------------+
|DEP_DELAY_Nulls|CARRIER_DELAY_Nulls|CANCELLATION_CODE|
+---------------+-------------------+-----------------+
|          27679|            1327781|          1629748|
+---------------+-------------------+-----------------+



In [24]:
# Find all flights where CANCELLATION_CODE = "B"
# (B = Weather cancellation)
# Count them.
# What % of total Q1 flights were weather cancelled?

a = df.filter(df["CANCELLATION_CODE"] == "B")
weather_cancelled = a.count()
total_flights = df.count()

print(f"Weather cancelled flights: {weather_cancelled:,}")
print(f"Total Q1 flights: {total_flights:,}")
print(f"Weather cancellation %: {(weather_cancelled/total_flights)*100:.2f}%")

Weather cancelled flights: 17,464
Total Q1 flights: 1,658,259
Weather cancellation %: 1.05%


In [25]:
# Find flights where:
#   → Origin state is "CA" (ORIGIN_STATE_ABR = "CA")
#   AND → ARR_DELAY is NOT null
#   AND → ARR_DELAY > 0
# Count total. Show 5 rows.
# Which CA airport appears most as origin?

b = df.filter((df["ORIGIN_STATE_ABR"] == "CA") & (df["ARR_DELAY"]>0) & (col("ARR_DELAY").isNotNull())).select(
    "ORIGIN", "DEST", "ARR_DELAY", "ORIGIN_STATE_ABR")
b.cache()
print(f"Total CA delayed flights: {b.count():,}")
b.show(5)
b.unpersist()

Total CA delayed flights: 60,420
+------+----+---------+----------------+
|ORIGIN|DEST|ARR_DELAY|ORIGIN_STATE_ABR|
+------+----+---------+----------------+
|   LAX| JFK|      8.0|              CA|
|   LAX| BOS|     19.0|              CA|
|   SNA| JFK|      8.0|              CA|
|   SFO| JFK|     33.0|              CA|
|   LAX| BOS|      8.0|              CA|
+------+----+---------+----------------+
only showing top 5 rows



DataFrame[ORIGIN: string, DEST: string, ARR_DELAY: double, ORIGIN_STATE_ABR: string]

**DAY_3**

In [26]:
# Create a new column called "DEP_DELAY_HOURS" 
# by dividing DEP_DELAY by 60.
# Show 5 rows with: ORIGIN, DEST, DEP_DELAY, DEP_DELAY_HOURS

a = df.withColumn("DEP_DELAY_HOURS", round(col("DEP_DELAY")/60,2)).select("ORIGIN", "DEST", "DEP_DELAY", "DEP_DELAY_HOURS")
a.show(5)

+------+----+---------+---------------+
|ORIGIN|DEST|DEP_DELAY|DEP_DELAY_HOURS|
+------+----+---------+---------------+
|   ATL| AVL|     -4.0|          -0.07|
|   DTW| OMA|     -4.0|          -0.07|
|   DSM| DTW|     -5.0|          -0.08|
|   LGA| SAV|     -5.0|          -0.08|
|   CHS| LGA|     -7.0|          -0.12|
+------+----+---------+---------------+
only showing top 5 rows



In [27]:
# Rename these 3 columns to cleaner names:
#   → OP_UNIQUE_CARRIER  to  CARRIER
#   → DEP_DELAY          to  DEPARTURE_DELAY
#   → ARR_DELAY          to  ARRIVAL_DELAY

# Show the new column names using df.columns

ps08 = df.withColumnRenamed("OP_UNIQUE_CARRIER", "CARRIER")\
    .withColumnRenamed("DEP_DELAY", "DEPARTURE_DELAY")\
    .withColumnRenamed("ARR_DELAY", "ARRIVAL_DELAY")

ps08.show(5)

+----+-----+------------+-----------+--------------------+-------+--------+-----------------+------+----------------+----------------+----+--------------+--------------+------------+--------+---------------+-------------+---------+------------+--------+-------------+-------------+---------+---------+-----------------+--------+----------------+-------------------+--------+-------+--------+-------------+-------------+---------+--------------+-------------------+
|YEAR|MONTH|DAY_OF_MONTH|DAY_OF_WEEK|             FL_DATE|CARRIER|TAIL_NUM|OP_CARRIER_FL_NUM|ORIGIN|ORIGIN_CITY_NAME|ORIGIN_STATE_ABR|DEST|DEST_CITY_NAME|DEST_STATE_ABR|CRS_DEP_TIME|DEP_TIME|DEPARTURE_DELAY|DEP_DELAY_NEW|DEP_DEL15|CRS_ARR_TIME|ARR_TIME|ARRIVAL_DELAY|ARR_DELAY_NEW|ARR_DEL15|CANCELLED|CANCELLATION_CODE|DIVERTED|CRS_ELAPSED_TIME|ACTUAL_ELAPSED_TIME|AIR_TIME|FLIGHTS|DISTANCE|CARRIER_DELAY|WEATHER_DELAY|NAS_DELAY|SECURITY_DELAY|LATE_AIRCRAFT_DELAY|
+----+-----+------------+-----------+--------------------+-------+----

In [28]:
# From df, drop these columns:
#   → TAIL_NUM
#   → OP_CARRIER_FL_NUM
#   → ORIGIN_STATE_ABR
#   → DEST_STATE_ABR
#   → DIVERTED

# Print column count before and after dropping.
# What columns remain?

print(f"Columns before dropping: {len(df.columns)}")
ps09 = df.drop("TAIL_NUM", "OP_CARRIER_FL_NUM", "ORIGIN_STATE_ABR", "DEST_STATE_ABR", "DIVERTED")
print(f"Columns After dropping: {len(ps09.columns)}")
print(ps09.columns)

Columns before dropping: 37
Columns After dropping: 32
['YEAR', 'MONTH', 'DAY_OF_MONTH', 'DAY_OF_WEEK', 'FL_DATE', 'OP_UNIQUE_CARRIER', 'ORIGIN', 'ORIGIN_CITY_NAME', 'DEST', 'DEST_CITY_NAME', 'CRS_DEP_TIME', 'DEP_TIME', 'DEP_DELAY', 'DEP_DELAY_NEW', 'DEP_DEL15', 'CRS_ARR_TIME', 'ARR_TIME', 'ARR_DELAY', 'ARR_DELAY_NEW', 'ARR_DEL15', 'CANCELLED', 'CANCELLATION_CODE', 'CRS_ELAPSED_TIME', 'ACTUAL_ELAPSED_TIME', 'AIR_TIME', 'FLIGHTS', 'DISTANCE', 'CARRIER_DELAY', 'WEATHER_DELAY', 'NAS_DELAY', 'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY']


**Practice_Problems**

In [29]:
# Create a new column "ARR_DELAY_HOURS" (ARR_DELAY / 60, rounded to 2)
# Then filter only flights where ARR_DELAY_HOURS > 1
# Show 5 rows with: ORIGIN, DEST, ARR_DELAY, ARR_DELAY_HOURS
# How many flights had arrival delay more than 1 hour?

p0301 = df.withColumn("ARR_DELAY_HOURS", round(col("ARR_DELAY")/60,2)).filter(col("ARR_DELAY_HOURS")>1)\
.select("ORIGIN", "DEST", "ARR_DELAY", "ARR_DELAY_HOURS")
p0301.cache()
p0301.show(5)
print(f"The flights had arrival delay more than 1 hour: {p0301.count():,} ")
p0301.unpersist()

+------+----+---------+---------------+
|ORIGIN|DEST|ARR_DELAY|ARR_DELAY_HOURS|
+------+----+---------+---------------+
|   AVL| ATL|    207.0|           3.45|
|   CLT| JFK|    162.0|            2.7|
|   RDU| LGA|    204.0|            3.4|
|   SAT| CLT|     78.0|            1.3|
|   BOS| LAX|     92.0|           1.53|
+------+----+---------+---------------+
only showing top 5 rows

The flights had arrival delay more than 1 hour: 110,406 


DataFrame[ORIGIN: string, DEST: string, ARR_DELAY: double, ARR_DELAY_HOURS: double]

In [30]:
# From df:
# 1. Drop: TAIL_NUM, DIVERTED, FLIGHTS
# 2. Rename: OP_UNIQUE_CARRIER → CARRIER
# 3. Select only: CARRIER, ORIGIN, DEST, DEP_DELAY, ARR_DELAY
# Show 5 rows.

p0302 = df.drop("TAIL_NUM", "DIVERTED", "FLIGHTS") \
          .withColumnRenamed("OP_UNIQUE_CARRIER", "CARRIER") \
          .select("CARRIER", "ORIGIN", "DEST", "DEP_DELAY", "ARR_DELAY")

p0302.show(5)

+-------+------+----+---------+---------+
|CARRIER|ORIGIN|DEST|DEP_DELAY|ARR_DELAY|
+-------+------+----+---------+---------+
|     9E|   ATL| AVL|     -4.0|    -17.0|
|     9E|   DTW| OMA|     -4.0|    -33.0|
|     9E|   DSM| DTW|     -5.0|     -8.0|
|     9E|   LGA| SAV|     -5.0|    -26.0|
|     9E|   CHS| LGA|     -7.0|    -21.0|
+-------+------+----+---------+---------+
only showing top 5 rows



In [31]:
# Create a new column "DELAY_STATUS":
#   → If DEP_DELAY <= 0   → "ON TIME"
#   → If DEP_DELAY <= 30  → "MINOR DELAY"
#   → If DEP_DELAY > 30   → "MAJOR DELAY"

# Show count of each status category.
# Hint: Use when/otherwise + groupBy


p0303 = df.withColumn("DELAY_STATUS",
    when(col("DEP_DELAY") <= 0, "ON TIME")
    .when(col("DEP_DELAY") <= 30, "MINOR DELAY")
    .otherwise("MAJOR DELAY")
)

p0303.groupBy("DELAY_STATUS").count().orderBy("count", ascending=False).show()

+------------+-------+
|DELAY_STATUS|  count|
+------------+-------+
|     ON TIME|1040495|
| MINOR DELAY| 383476|
| MAJOR DELAY| 234288|
+------------+-------+



In [32]:
# 1. Count nulls in DEP_DELAY
# 2. Fill nulls in DEP_DELAY with 0
# 3. On the filled DataFrame, create a new column 
#    "IS_EARLY" = True if DEP_DELAY < 0, else False
# 4. Count how many IS_EARLY = True

# Step 1 - Count nulls in DEP_DELAY
null_count = df.filter(col("DEP_DELAY").isNull()).count()
print(f"Nulls in DEP_DELAY: {null_count:,}")

# Step 2 - Fill nulls with 0
df_filled = df.fillna(0, subset=["DEP_DELAY"])

# Step 3 - Create IS_EARLY column
p0304 = df_filled.withColumn("IS_EARLY",
    when(col("DEP_DELAY") < 0, True).otherwise(False)
)

# Step 4 - Count IS_EARLY = True
early_count = p0304.filter(col("IS_EARLY") == True).count()
print(f"Early departures: {early_count:,}")

Nulls in DEP_DELAY: 27,679
Early departures: 968,879


In [33]:
# Build this in ONE chain (no intermediate variables):
#   → Filter: ORIGIN = "ORD" (Chicago O'Hare)
#   → Drop: TAIL_NUM, DIVERTED, FLIGHTS, CANCELLATION_CODE
#   → Rename: OP_UNIQUE_CARRIER → CARRIER
#   → Create new column: DEP_DELAY_HOURS = DEP_DELAY / 60 (round 2)
#   → Select: CARRIER, ORIGIN, DEST, DEP_DELAY, DEP_DELAY_HOURS
#   → Show 5 rows

p0305 = df.filter(df["ORIGIN"] == "ORD") \
          .drop("TAIL_NUM", "DIVERTED", "FLIGHTS", "CANCELLATION_CODE") \
          .withColumnRenamed("OP_UNIQUE_CARRIER", "CARRIER") \
          .withColumn("DEP_DELAY_HOURS", round(col("DEP_DELAY") / 60, 2)) \
          .select("CARRIER", "ORIGIN", "DEST", "DEP_DELAY", "DEP_DELAY_HOURS")

p0305.show(5)

+-------+------+----+---------+---------------+
|CARRIER|ORIGIN|DEST|DEP_DELAY|DEP_DELAY_HOURS|
+-------+------+----+---------+---------------+
|     AA|   ORD| DCA|     -4.0|          -0.07|
|     AA|   ORD| DFW|      1.0|           0.02|
|     AA|   ORD| DFW|     -4.0|          -0.07|
|     AA|   ORD| PHX|     24.0|            0.4|
|     AA|   ORD| PHX|      5.0|           0.08|
+-------+------+----+---------+---------------+
only showing top 5 rows



**DAY_4** 

In [34]:
# 1. Find all unique carriers in the dataset
#    How many unique carriers operated in Q1 2024?

ps10 = df.select("OP_UNIQUE_CARRIER").distinct()
ps10.show(5)
print(f"Total unique carriers: {ps10.count()}")

# 2. Find all unique ORIGIN states (ORIGIN_STATE_ABR)
#    How many unique states did flights originate from?

ps10_1 = df.select("ORIGIN_STATE_ABR").distinct()
ps10_1.show(5)
print(f"Total Unique Origin States : {ps10_1.count()}")

+-----------------+
|OP_UNIQUE_CARRIER|
+-----------------+
|               UA|
|               NK|
|               AA|
|               B6|
|               DL|
+-----------------+
only showing top 5 rows

Total unique carriers: 15
+----------------+
|ORIGIN_STATE_ABR|
+----------------+
|              SC|
|              AZ|
|              LA|
|              MN|
|              NJ|
+----------------+
only showing top 5 rows

Total Unique Origin States : 52


In [35]:
# Find average ARR_DELAY per carrier.
# Sort results:
#   → First by avg_arr_delay DESCENDING (worst carriers first)
#   → Then show same result ASCENDING (best carriers first)
# Which carrier has the worst average arrival delay?
# Which carrier has the best?

from pyspark.sql.functions import avg, round

ps11 = df.groupBy("OP_UNIQUE_CARRIER").agg(round(avg("ARR_DELAY"),2).alias("avg_arr_delay"))
print("Worst Carriers:")
ps11.orderBy("avg_arr_delay",ascending= False).show(5)
print("Best Carriers:")
ps11.orderBy("avg_arr_delay",Ascending= True).show(5)


Worst Carriers:
+-----------------+-------------+
|OP_UNIQUE_CARRIER|avg_arr_delay|
+-----------------+-------------+
|               F9|        13.87|
|               AA|        13.64|
|               B6|        13.08|
|               NK|         9.26|
|               G4|         8.95|
+-----------------+-------------+
only showing top 5 rows

Best Carriers:
+-----------------+-------------+
|OP_UNIQUE_CARRIER|avg_arr_delay|
+-----------------+-------------+
|               YX|        -6.52|
|               9E|         0.14|
|               DL|         0.82|
|               UA|         4.04|
|               AS|         4.24|
+-----------------+-------------+
only showing top 5 rows



In [36]:
# From the full df:
#   → Select: ORIGIN, DEST, DEP_DELAY, ARR_DELAY
#   → Order by DEP_DELAY descending
#   → Get only TOP 10 worst departure delays
# What was the single worst departure delay in Q1 2024?
# Which route was it?

ps12 = df.select("ORIGIN", "DEST", "DEP_DELAY", "ARR_DELAY").orderBy(col("DEP_DELAY").desc_nulls_last(), Ascending = False).show(10)

+------+----+---------+---------+
|ORIGIN|DEST|DEP_DELAY|ARR_DELAY|
+------+----+---------+---------+
|   SEA| CLT|   3360.0|   3359.0|
|   IAH| DFW|   3125.0|   3136.0|
|   BNA| DFW|   2972.0|   2989.0|
|   PWM| CLT|   2923.0|   2901.0|
|   TYS| CLT|   2892.0|   2884.0|
|   STX| MIA|   2891.0|   2899.0|
|   SFO| DFW|   2816.0|   2795.0|
|   DSM| DFW|   2809.0|   2833.0|
|   PIT| PHL|   2800.0|   2779.0|
|   BNA| DFW|   2696.0|   2700.0|
+------+----+---------+---------+
only showing top 10 rows



**Practice_Problems**

In [37]:
df.columns

['YEAR',
 'MONTH',
 'DAY_OF_MONTH',
 'DAY_OF_WEEK',
 'FL_DATE',
 'OP_UNIQUE_CARRIER',
 'TAIL_NUM',
 'OP_CARRIER_FL_NUM',
 'ORIGIN',
 'ORIGIN_CITY_NAME',
 'ORIGIN_STATE_ABR',
 'DEST',
 'DEST_CITY_NAME',
 'DEST_STATE_ABR',
 'CRS_DEP_TIME',
 'DEP_TIME',
 'DEP_DELAY',
 'DEP_DELAY_NEW',
 'DEP_DEL15',
 'CRS_ARR_TIME',
 'ARR_TIME',
 'ARR_DELAY',
 'ARR_DELAY_NEW',
 'ARR_DEL15',
 'CANCELLED',
 'CANCELLATION_CODE',
 'DIVERTED',
 'CRS_ELAPSED_TIME',
 'ACTUAL_ELAPSED_TIME',
 'AIR_TIME',
 'FLIGHTS',
 'DISTANCE',
 'CARRIER_DELAY',
 'WEATHER_DELAY',
 'NAS_DELAY',
 'SECURITY_DELAY',
 'LATE_AIRCRAFT_DELAY']

In [38]:
# Find all unique DEST airports that received 
# flights from "ORD" (Chicago O'Hare)
# How many unique destinations does ORD serve in Q1 2024?

ps41 = df.where(df["ORIGIN"] == "ORD").select("DEST").distinct()
ps41.show(5)
print(f"The count is: {ps41.count():} ")

+----+
|DEST|
+----+
| MSY|
| GEG|
| SNA|
| GRB|
| FOD|
+----+
only showing top 5 rows

The count is: 147 


In [39]:
# Create a new column "SPEED" = DISTANCE / AIR_TIME
# (miles per minute — rough airspeed estimate)
# Show TOP 10 fastest flights
# Columns: ORIGIN, DEST, DISTANCE, AIR_TIME, SPEED

ps42 = df.withColumn("SPEED", col("DISTANCE")/col("AIR_TIME")).select("ORIGIN", "DEST", "DISTANCE", "AIR_TIME", "SPEED").orderBy("SPEED",ascending=False)\
    .show(10)

+------+----+--------+--------+------------------+
|ORIGIN|DEST|DISTANCE|AIR_TIME|             SPEED|
+------+----+--------+--------+------------------+
|   MEM| SFB|   671.0|     9.0| 74.55555555555556|
|   ALB| MYR|   685.0|    40.0|            17.125|
|   DEN| MCO|  1546.0|   116.0|13.327586206896552|
|   DEN| MIA|  1709.0|   130.0|13.146153846153846|
|   MCO| PSE|  1179.0|    92.0|12.815217391304348|
|   DTW| JFK|   509.0|    40.0|            12.725|
|   DTW| TPA|   983.0|    78.0|12.602564102564102|
|   SMF| LAX|   373.0|    31.0| 12.03225806451613|
|   MCO| PSE|  1179.0|    98.0| 12.03061224489796|
|   MIA| BOS|  1258.0|   105.0|11.980952380952381|
+------+----+--------+--------+------------------+
only showing top 10 rows



In [40]:
# Find all unique carriers that flew OUT of "JFK"
# Order them alphabetically (ascending)
# Show only first 5

ps43 = df.filter(df["ORIGIN"] == "JFK").select("OP_UNIQUE_CARRIER").distinct().orderBy("OP_UNIQUE_CARRIER",ascending = True).show(5)

+-----------------+
|OP_UNIQUE_CARRIER|
+-----------------+
|               9E|
|               AA|
|               AS|
|               B6|
|               DL|
+-----------------+
only showing top 5 rows



In [41]:
# Find flights where CARRIER_DELAY is NOT null
# Order by CARRIER_DELAY descending (nulls last)
# Show top 5
# Which carrier caused the longest carrier delay?

ps44 = df.filter(col("CARRIER_DELAY").isNotNull()).orderBy(col("CARRIER_DELAY").desc_nulls_last())\
    .select("OP_UNIQUE_CARRIER", "ORIGIN", "DEST", "CARRIER_DELAY").show(5)

+-----------------+------+----+-------------+
|OP_UNIQUE_CARRIER|ORIGIN|DEST|CARRIER_DELAY|
+-----------------+------+----+-------------+
|               AA|   SEA| CLT|       3359.0|
|               AA|   BNA| DFW|       2972.0|
|               AA|   PWM| CLT|       2901.0|
|               AA|   TYS| CLT|       2884.0|
|               AA|   IAH| DFW|       2825.0|
+-----------------+------+----+-------------+
only showing top 5 rows



In [42]:
# Build in ONE chain:
#   → Filter: MONTH = 1 (January only)
#   → Filter: CANCELLED = 0 (completed flights only)
#   → Drop: CANCELLATION_CODE, DIVERTED, TAIL_NUM
#   → Create: DELAY_DIFF = ARR_DELAY - DEP_DELAY
#     (positive = got worse in air, negative = made up time)
#   → Select: ORIGIN, DEST, DEP_DELAY, ARR_DELAY, DELAY_DIFF
#   → Order by DELAY_DIFF descending (nulls last)
#   → Show top 10

ps45 = df.filter((df["MONTH"] == 1) & (df["CANCELLED"] == 0)).drop("CANCELLATION_CODE", "DIVERTED", "TAIL_NUM")\
    .withColumn("DELAY_DIFF", col("ARR_DELAY") - col("DEP_DELAY")).select("ORIGIN", "DEST", "DEP_DELAY", "ARR_DELAY", "DELAY_DIFF")\
        .orderBy(col("DELAY_DIFF").desc_nulls_last()).show(10)

+------+----+---------+---------+----------+
|ORIGIN|DEST|DEP_DELAY|ARR_DELAY|DELAY_DIFF|
+------+----+---------+---------+----------+
|   PHX| LAS|    -15.0|    639.0|     654.0|
|   DCA| BNA|     -9.0|    491.0|     500.0|
|   PBI| PHL|    167.0|    585.0|     418.0|
|   LGA| BNA|     38.0|    385.0|     347.0|
|   ORD| BNA|    101.0|    444.0|     343.0|
|   MSN| LGA|    250.0|    554.0|     304.0|
|   MCO| DCA|     99.0|    391.0|     292.0|
|   PHL| BNA|     47.0|    312.0|     265.0|
|   MCI| DCA|    301.0|    560.0|     259.0|
|   ORD| BNA|     50.0|    302.0|     252.0|
+------+----+---------+---------+----------+
only showing top 10 rows



**DAY_5**

In [49]:
# 1. Write your existing df to Parquet format first:
#    df.write.parquet("../data/parquet/bts_q1_2024")

#df.write.mode("overwrite").parquet(r"C:\Users\shivn\Downloads\Practice Data 3 months\parquet\bts_q1_2024")

# 3. Compare:
#    → How long does reading Parquet take vs CSV?
#    → Print schema of both — what's different?
#    → Count rows — same number?

import time

# CSV read time
start = time.time()
df_csv = spark.read.csv(r"C:\Users\shivn\Downloads\Practice Data 3 months\January_2024.csv", header=True, inferSchema=True)
df_csv.count()
csv_time = time.time() - start
print(f"CSV read time: {csv_time:.2f} seconds")

# Parquet read time
start = time.time()
df_parquet = spark.read.parquet(r"C:\Users\shivn\Downloads\Practice Data 3 months\parquet\bts_q1_2024")
df_parquet.count()
parquet_time = time.time() - start
print(f"Parquet read time: {parquet_time:.2f} seconds")

CSV read time: 0.62 seconds
Parquet read time: 0.17 seconds


In [47]:
#    spark.read.parquet("../data/parquet/bts_q1_2024")
df_parquet = spark.read.parquet(r"C:\Users\shivn\Downloads\Practice Data 3 months\parquet\bts_q1_2024")

df_parquet.printSchema()
print(f"Row count: {df_parquet.count():,}")

root
 |-- YEAR: integer (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- DAY_OF_MONTH: integer (nullable = true)
 |-- DAY_OF_WEEK: integer (nullable = true)
 |-- FL_DATE: string (nullable = true)
 |-- OP_UNIQUE_CARRIER: string (nullable = true)
 |-- TAIL_NUM: string (nullable = true)
 |-- OP_CARRIER_FL_NUM: integer (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- ORIGIN_CITY_NAME: string (nullable = true)
 |-- ORIGIN_STATE_ABR: string (nullable = true)
 |-- DEST: string (nullable = true)
 |-- DEST_CITY_NAME: string (nullable = true)
 |-- DEST_STATE_ABR: string (nullable = true)
 |-- CRS_DEP_TIME: integer (nullable = true)
 |-- DEP_TIME: integer (nullable = true)
 |-- DEP_DELAY: double (nullable = true)
 |-- DEP_DELAY_NEW: double (nullable = true)
 |-- DEP_DEL15: double (nullable = true)
 |-- CRS_ARR_TIME: integer (nullable = true)
 |-- ARR_TIME: integer (nullable = true)
 |-- ARR_DELAY: double (nullable = true)
 |-- ARR_DELAY_NEW: double (nullable = true)
 |-- A

In [50]:
# Write df to Parquet partitioned by MONTH:
#    .partitionBy("MONTH")

# Save to: "../data/parquet/bts_partitioned"
# After writing, check your file explorer —
# what folders were created inside?

df.write.mode("overwrite").partitionBy("MONTH") \
   .parquet(r"C:\Users\shivn\Downloads\Practice Data 3 months\parquet\bts_partitioned")

In [51]:
# Read back the partitioned Parquet.
# 1. Read only MONTH=1 partition
# 2. Count rows — should match January count (547,271)
# 3. Print schema — is MONTH column still there?

df_jan = spark.read.parquet(
    r"C:\Users\shivn\Downloads\Practice Data 3 months\parquet\bts_partitioned"
).filter("MONTH = 1")

print(f"January row count: {df_jan.count():,}")
df_jan.printSchema()

January row count: 547,271
root
 |-- YEAR: integer (nullable = true)
 |-- DAY_OF_MONTH: integer (nullable = true)
 |-- DAY_OF_WEEK: integer (nullable = true)
 |-- FL_DATE: string (nullable = true)
 |-- OP_UNIQUE_CARRIER: string (nullable = true)
 |-- TAIL_NUM: string (nullable = true)
 |-- OP_CARRIER_FL_NUM: integer (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- ORIGIN_CITY_NAME: string (nullable = true)
 |-- ORIGIN_STATE_ABR: string (nullable = true)
 |-- DEST: string (nullable = true)
 |-- DEST_CITY_NAME: string (nullable = true)
 |-- DEST_STATE_ABR: string (nullable = true)
 |-- CRS_DEP_TIME: integer (nullable = true)
 |-- DEP_TIME: integer (nullable = true)
 |-- DEP_DELAY: double (nullable = true)
 |-- DEP_DELAY_NEW: double (nullable = true)
 |-- DEP_DEL15: double (nullable = true)
 |-- CRS_ARR_TIME: integer (nullable = true)
 |-- ARR_TIME: integer (nullable = true)
 |-- ARR_DELAY: double (nullable = true)
 |-- ARR_DELAY_NEW: double (nullable = true)
 |-- ARR_DEL15: d

**Practice_Problems**